# FarmTech Solutions — treinamento agrícola no Colab
Selecione runtime com GPU. Este notebook inicia novos treinamentos reais de 30 e 60 épocas.
Envie FarmTech.zip, gerado a partir do projeto integrado. Nenhum resultado prévio
de garrafas/mochilas é usado como evidência agrícola. Veja docs/colab.md.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, sys, subprocess, os
uploaded = files.upload()
archive = next(Path(name) for name in uploaded if name.endswith('.zip'))
target = Path('/content/farmtech_project')
if target.exists():
    raise FileExistsError('Pasta já existe: preserve o experimento ou use novo runtime')
with zipfile.ZipFile(archive) as package:
    for info in package.infolist():
        path = (target / info.filename).resolve()
        if not path.is_relative_to(target.resolve()) or (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('ZIP contém caminho inseguro')
    package.extractall(target)
ROOT = target / 'FarmTech'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use runtime Python 3.11/3.12 compatível com PyTorch 2.5.1')


In [ ]:
# Instalação explícita: execute uma vez e reinicie a sessão quando solicitado.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements/visao.txt', '-c', 'requirements/constraints-colab.txt'], check=True)
runtime = ROOT / '.runtime/yolov5'
runtime.parent.mkdir(exist_ok=True)
if not runtime.exists():
    subprocess.run(['git', 'clone', '--branch', 'v7.0', '--depth', '1', 'https://github.com/ultralytics/yolov5.git', str(runtime)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(runtime / 'requirements.txt'), '-c', 'requirements/constraints-colab.txt'], check=True)
os.environ['YOLOv5_AUTOINSTALL'] = 'false'


## Preparar dados e revisar anotações
O download abaixo usa o repositório dos autores. A conversão filtra tomate,
agrupa sete doenças e preserva o teste oficial. Examine o manifesto, candidatos
a duplicatas e amostras de cada classe antes de iniciar o treinamento.


In [ ]:
source = Path('/content/PlantDoc')
if not source.exists():
    from farmtech.services.visao.fetch import fetch
    print(fetch(source))
dataset = Path('/content/tomate_dataset')
from farmtech.services.visao.dataset import prepare
manifest = prepare(source, dataset)
print('Fingerprint:', manifest['fingerprint'])
from collections import Counter
print(Counter(row['split'] for row in manifest['imagens']))
print('Inspecione /content/tomate_dataset/manifest.json e as imagens/labels antes de avançar.')


## Persistir no Drive e treinar
A GPU executa os dois treinamentos. Checkpoints e resultados ficam em uma pasta
nova no Drive, para sobreviver ao encerramento do runtime. A seleção é por validação;
o teste é executado somente no checkpoint selecionado. Não ajuste parâmetros pelo teste.


In [ ]:
from google.colab import drive
from datetime import datetime, timezone
drive.mount('/content/drive')
session = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
destination = Path('/content/drive/MyDrive/Faculdade/Fiap/Fase7') / session
from farmtech.services.visao.train import train
result = train(dataset, runtime, destination)


## Exportar e interpretar
O ZIP contém pesos, versões, hash e métricas reais. Copie o conteúdo para models/tomate
na EC2. Preserve runs e faça revisão de acertos e erros nas imagens do teste.
Se uma etapa falhar, registre a falha; não preencha métricas manualmente.


In [ ]:
files.download(str(destination / 'tomate_bundle.zip'))